In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root / "src"))

### Testing ReLU


In [14]:
from tinytorch.activations import ReLU_CP
from torch.nn import ReLU
from tinytorch.tensor import Tensor_CP
from torch import Tensor
import numpy as np

In [15]:
x = Tensor(np.random.randn(3, 4) * 5)
x.data

tensor([[ 1.9819, -2.8543,  4.1708, -0.2937],
        [-0.4303, -4.0570,  5.1613, -3.6049],
        [-0.9169, -2.1263, -1.9871,  5.2484]])

In [16]:
relu = ReLU()
y = relu(x)

print("Input x:")
print(x.data)

print("\nAfter ReLU:")
print(y.data)

Input x:
tensor([[ 1.9819, -2.8543,  4.1708, -0.2937],
        [-0.4303, -4.0570,  5.1613, -3.6049],
        [-0.9169, -2.1263, -1.9871,  5.2484]])

After ReLU:
tensor([[1.9819, 0.0000, 4.1708, 0.0000],
        [0.0000, 0.0000, 5.1613, 0.0000],
        [0.0000, 0.0000, 0.0000, 5.2484]])


### Manual ReLU

In [17]:
x = Tensor_CP(np.random.randn(3, 4) * 5)
x.data

array([[ 2.6529796 ,  0.8754932 , -4.979886  ,  0.46333012],
       [-7.466921  , -0.24971627, -4.4933734 , -8.673383  ],
       [ 3.8841102 , -3.553379  ,  3.3349288 ,  2.650829  ]],
      dtype=float32)

In [18]:
relu = ReLU_CP()
y = relu(x)

print("Input x:")
print(x.data)

print("\nAfter ReLU:")
print(y.data)

Input x:
[[ 2.6529796   0.8754932  -4.979886    0.46333012]
 [-7.466921   -0.24971627 -4.4933734  -8.673383  ]
 [ 3.8841102  -3.553379    3.3349288   2.650829  ]]

After ReLU:
[[2.6529796  0.8754932  0.         0.46333012]
 [0.         0.         0.         0.        ]
 [3.8841102  0.         3.3349288  2.650829  ]]


### Testing Sigmoid

In [19]:
from tinytorch.activations import Sigmoid_CP
from torch.nn import Sigmoid

# Same underlying data for both, so the comparison is fair
x_np = np.random.randn(3, 4) * 5

# PyTorch reference
y_torch = Sigmoid()(Tensor(x_np))

# Manual implementation
y_cp = Sigmoid_CP()(Tensor_CP(x_np))

print("PyTorch Sigmoid:\n", y_torch.numpy())
print("\nManual Sigmoid:\n", y_cp.data)

assert np.allclose(y_torch.numpy(), y_cp.data, atol=1e-5), "Sigmoid mismatch!"
print("\nSigmoid matches PyTorch")

PyTorch Sigmoid:
 [[0.8471025  0.9621274  0.99990225 0.99999416]
 [0.25093022 0.9998895  0.9220261  0.00176558]
 [0.9361206  0.9980373  0.9751024  0.75707567]]

Manual Sigmoid:
 [[0.8471025  0.9621274  0.99990225 0.99999416]
 [0.25093025 0.9998895  0.9220261  0.00176558]
 [0.9361206  0.9980373  0.9751024  0.7570756 ]]

Sigmoid matches PyTorch


### Testing Tanh

In [20]:
from tinytorch.activations import Tanh_CP
from torch.nn import Tanh

x_np = np.random.randn(3, 4) * 5

# PyTorch reference
y_torch = Tanh()(Tensor(x_np))

# Manual implementation
y_cp = Tanh_CP()(Tensor_CP(x_np))

print("PyTorch Tanh:\n", y_torch.numpy())
print("\nManual Tanh:\n", y_cp.data)

assert np.allclose(y_torch.numpy(), y_cp.data, atol=1e-5), "Tanh mismatch!"
print("\nTanh matches PyTorch")

PyTorch Tanh:
 [[-0.79922897  0.9999951  -0.99905044  0.99562156]
 [ 0.93892705  0.96592164 -1.         -0.9989721 ]
 [-0.41910765  1.          0.99999803  0.9999754 ]]

Manual Tanh:
 [[-0.79922897  0.9999951  -0.99905044  0.99562156]
 [ 0.938927    0.96592164 -0.99999994 -0.9989721 ]
 [-0.41910765  1.          0.99999803  0.99997544]]

Tanh matches PyTorch


### Testing GeLU

`GeLU_CP` uses the **sigmoid approximation** $x \cdot \sigma(1.702x)$, while PyTorch's default `GELU()` is the exact erf-based version. They differ slightly, so we compare with a looser tolerance.

Note: `GeLU_CP` currently has no `__call__`, so we call `.forward()` directly.

In [21]:
from tinytorch.activations import GeLU_CP
from torch.nn import GELU

x_np = np.random.randn(3, 4) * 5

# PyTorch reference (exact, erf-based)
y_torch = GELU()(Tensor(x_np))

# Manual implementation (sigmoid approximation) -- no __call__, use .forward()
y_cp = GeLU_CP().forward(Tensor_CP(x_np))

print("PyTorch GELU (exact):\n", y_torch.numpy())
print("\nManual GELU (sigmoid approx):\n", y_cp.data)

max_diff = np.max(np.abs(y_torch.numpy() - y_cp.data))
print(f"\nMax abs difference vs exact GELU: {max_diff:.4f}")

assert np.allclose(y_torch.numpy(), y_cp.data, atol=0.05), "GELU differs beyond approximation tolerance!"
print("GELU matches PyTorch within approximation tolerance")

PyTorch GELU (exact):
 [[ 8.7844429e+00 -4.7074119e-04  0.0000000e+00 -3.5278322e-03]
 [ 1.1168177e+00 -1.4367305e-01  1.6534832e+01  0.0000000e+00]
 [ 8.9047365e+00  6.7930942e+00 -1.6268096e-07 -4.3777563e-03]]

Manual GELU (sigmoid approx):
 [[ 8.7844400e+00 -7.2538760e-03 -4.5925462e-05 -1.6964843e-02]
 [ 1.1158410e+00 -1.3986772e-01  1.6534832e+01 -3.8307626e-05]
 [ 8.9047346e+00  6.7930293e+00 -5.0369487e-04 -1.8733228e-02]]

Max abs difference vs exact GELU: 0.0144
GELU matches PyTorch within approximation tolerance


### Testing Softmax

In [22]:
from tinytorch.activations import Softmax_CP
from torch.nn import Softmax

x_np = np.random.randn(3, 4) * 5

# PyTorch reference
y_torch = Softmax(dim=-1)(Tensor(x_np))

# Manual implementation
y_cp = Softmax_CP()(Tensor_CP(x_np), dim=-1)

print("PyTorch Softmax:\n", y_torch.numpy())
print("\nManual Softmax:\n", y_cp.data)

# Each row should be a valid probability distribution (sums to 1)
print("\nRow sums (manual):", y_cp.data.sum(axis=-1))

assert np.allclose(y_torch.numpy(), y_cp.data, atol=1e-5), "Softmax mismatch!"
assert np.allclose(y_cp.data.sum(axis=-1), 1.0, atol=1e-5), "Softmax rows don't sum to 1!"
print("\nSoftmax matches PyTorch and rows sum to 1")

PyTorch Softmax:
 [[2.6644889e-01 6.6609651e-01 1.7532969e-02 4.9921587e-02]
 [1.7635049e-01 6.2147374e-03 3.5982674e-01 4.5760798e-01]
 [8.7210769e-03 3.5701330e-05 9.4743985e-01 4.3803409e-02]]

Manual Softmax:
 [[2.6644889e-01 6.6609651e-01 1.7532969e-02 4.9921587e-02]
 [1.7635052e-01 6.2147374e-03 3.5982674e-01 4.5760798e-01]
 [8.7210778e-03 3.5701330e-05 9.4743985e-01 4.3803405e-02]]

Row sums (manual): [0.99999994 1.         1.        ]

Softmax matches PyTorch and rows sum to 1
